<a href="https://colab.research.google.com/github/MawaSC/Quantum-Simulation-on-IQM-Resonance/blob/main/06_error_mitigation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit

from fermi_hubbard import (
    L,
    N_Q,
    init_occ,
    build_circuit,
    occupations_from_counts,
    qubit_of
)

1. Readout-error mitigation

We run calibration circuits to check how often each qubit is measured as 1 when it was prepared in 0, and how often it is measured as 1 when it was prepared in 1.

We use these results to correct the measured occupation probabilities for each qubit. This method corrects each qubit separately, so it does not account for every possible error involving multiple qubits.

We compare the results before and after correction against the same reference. We then check whether the correction actually reduces the error, since mitigation does not always improve the results.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

dt = 0.10
n_steps = int(3.0 / dt)

shots = 4096

print(f"dt = {dt}")
print(f"Trotter steps = {n_steps}")
print(f"Total simulation time = {dt * n_steps}")
print(f"Shots = {shots}")

In [ ]:
qv = build_circuit(
    dt=dt,
    n_steps=n_steps,
    init_occ=init_occ
)

qc = qv.qs.to_qiskit()
qc.measure_all()

print("Circuit depth:", qc.depth())
print("Number of gates:", len(qc.data))

In [ ]:
#Run Ideal Simulation

ideal_result = ideal_sim.run(
    qc,
    shots=shots,
    seed_simulator=1234
).result()

ideal_counts = ideal_result.get_counts()

ideal_up, ideal_down = occupations_from_counts(
    ideal_counts,
    flipped=(n_steps % 2 == 1)
)

ideal_total = ideal_up + ideal_down

print("Ideal total occupation:")
print(ideal_total)

In [ ]:
#Depolarizing Noise Simulation

seeds = [1234, 5678, 9012, 3456, 7890]

noisy_total_runs = []

for seed in seeds:

    noisy_result = noisy_sim.run(
        qc,
        shots=shots,
        seed_simulator=seed
    ).result()

    noisy_counts = noisy_result.get_counts()

    noisy_up, noisy_down = occupations_from_counts(
        noisy_counts,
        flipped=(n_steps % 2 == 1)
    )

    noisy_total_runs.append(noisy_up + noisy_down)

#noisy_total_runs = np.mean(noisy_total_runs)

noisy_total_runs = np.array(noisy_total_runs)

noisy_mean = np.mean(noisy_total_runs, axis=0)
noisy_std = np.std(noisy_total_runs, axis=0, ddof=1)

print("Noisy mean:")
print(noisy_mean)
print("\nNoisy standard deviation:")
print(noisy_std)

print("Depolarizing Noise mean occupation:")
print(noisy_mean)

print("Depolarizing noise standard deviation:")
print(noisy_std)

In [ ]:
#Calibration Circuits

from qiskit import QuantumCircuit

def create_readout_calibration_circuits(n_qubits):

  circuits = []

  #Prepare |000...0>
  qc0 = QuantumCircuit(n_qubits, n_qubits)

  for q in range(n_qubits):
    qc0.measure(q, q)

  circuits.append(qc0)

  #Prepare |111...1>
  qc1 = QuantumCircuit(n_qubits, n_qubits)

  for q in range(n_qubits):
    qc1.x(q)

  for q in range(n_qubits):
    qc1.measure(q, q)

  circuits.append(qc1)

  return circuits

In [ ]:
#Run Calibration on Noisy Simulator

calibration_circuits = create_readout_calibration_circuits(N_Q)

calibration_results = noisy_sim.run(
    calibration_circuits,
    shots=shots,
    seed_simulator=1234
).result()

cal_zero_counts = calibration_results.get_counts(0)
cal_one_counts = calibration_results.get_counts(1)

print("Calibration |000...0|:")
print(cal_zero_counts)

print("\nCalibration |111...1|:")
print(cal_one_counts)

In [ ]:
#Estimate Readout Errors

def get_bit_probabilities(counts, n_qubits):

  probabilities = np.zeros(n_qubits)

  total_shots = sum(counts.values())

  for bitstring, count in counts.items():

    probability = count / total_shots

    #Reverse Qiskit bitstrong ordering
    bitstring = bitstring[::-1]

    for q in range(n_qubits):

      if bitstring[q] == "1":
        probabilities[q] += probability

  return probabilities

In [ ]:
p1_when_zero = get_bit_probabilities(cal_zero_counts, N_Q)
p1_when_one = get_bit_probabilities(cal_one_counts, N_Q)

print("P(measured 1 | prepared 0):")
print(p1_when_zero)
print("\nP(measured 1 | prepared 1):")
print(p1_when_one)

In [ ]:
#Readout Error Mitigation

def mitigate_readout_probabilities(measured_probabilities, p1_when_zero, p1_when_one):

  corrected = np.zeros(len(measured_probabilities))

  for q in range(len(measured_probabilities)):
    a = p1_when_zero[q]
    b = p1_when_one[q]

    denominator = b - a

    if abs(denominator) < 1e-12:
      corrected[q] = measured_probabilities[q]
    else:
      corrected[q] = (measured_probabilities[q] - a) / denominator

  return np.clip(corrected, 0, 1)

In [ ]:
#Measure Noisy Circuit

noisy_result = noisy_sim.run(
    qc,
    shots=shots,
    seed_simulator=1234
).result()

noisy_counts = noisy_result.get_counts()

measured_probabilities = get_bit_probabilities(
    noisy_counts,
    N_Q
)

mitigated_probabilities = mitigate_readout_probabilities(
    measured_probabilities,
    p1_when_zero,
    p1_when_one
)

print("Measured probabilities:")
print(measured_probabilities)

print("\nMitigated probabilities:")
print(mitigated_probabilities)

In [ ]:
#Convert Mitigated Qubit Proabilities to Site Occupations

mitigated_up = np.array([
    mitigated_probabilities[
        qubit_of(i, 'u')
    ]
    for i in range(L)
])

mitigated_down = np.array([
    mitigated_probabilities[
        qubit_of(i, 'd')
    ]
    for i in range(L)
])

mitigated_total = mitigated_up + mitigated_down

print("Mitigated spin-up:")
print(mitigated_up)

print("\nMitigated spin-down:")
print(mitigated_down)

print("\nMitigated total occupation:")
print(mitigated_total)

In [ ]:
#RMSE Comparison

rmse_noisy = np.sqrt(
    np.mean(
        (noisy_mean - ideal_total) ** 2
    )
)

rmse_mitigated = np.sqrt(
    np.mean(
        (mitigated_total - ideal_total) ** 2
    )
)

rmse_hardware = np.sqrt(
    np.mean(
        (total_hardware - ideal_total) ** 2
    )
)

print(f"RMSE - Depolarizing noise: {rmse_noisy:.5f}")
print(f"RMSE - Mitigated:           {rmse_mitigated:.5f}")
print(f"RMSE - Garnet hardware:     {rmse_hardware:.5f}")

In [ ]:
#Error Reduction

if rmse_noisy > 0:

    improvement = (
        (rmse_noisy - rmse_mitigated)
        / rmse_noisy
    ) * 100

    print(f"Error reduction from mitigation: {improvement:.2f}%")

In [ ]:
#Comparison Table

comparison = pd.DataFrame({

    "Site": np.arange(L),

    "Ideal": ideal_total,

    "Depolarizing Noise": noisy_mean,

    "Mitigated": mitigated_total,

    "IQM Garnet": total_hardware
})

comparison = comparison.round(3)

print(comparison)

In [ ]:
#Plots

sites = np.arange(L)

plt.figure(figsize=(9, 5))

plt.plot(
    sites,
    ideal_total,
    'o-',
    label="Ideal"
)

plt.plot(
    sites,
    noisy_mean,
    's--',
    label="Depolarizing Noise"
)

plt.plot(
    sites,
    mitigated_total,
    'D--',
    label="Mitigated"
)

plt.plot(
    sites,
    total_hardware,
    '^--',
    label="IQM Garnet"
)

plt.xlabel("Lattice Site")
plt.ylabel("Total Occupation")

plt.title(
    "Fermi-Hubbard Site Occupation: "
    "Ideal, Noisy, Mitigated, and Hardware"
)

plt.xticks(sites)
plt.grid()
plt.legend()

plt.show()